In [161]:
import numpy as np
import numba as nb



@nb.njit
def weighted_running_average_naive(arr, weights):
    out = np.full(arr.shape, np.nan)
    for i in range(len(arr)):
        out[i] = np.sum(arr[:i + 1] * weights[:i + 1]) / np.sum(weights[:i + 1])
    return out, None, None


@nb.njit
def weighted_moving_average_naive(arr, weights, window_width):
    out = np.full(arr.shape, np.nan)
    out[:window_width] = weighted_running_average_naive(arr[:window_width], weights[:window_width])[0]
    for i in range(window_width, len(arr)):
        out[i] = np.sum(arr[i - window_width + 1:i + 1] * weights[i - window_width + 1:i + 1]) / np.sum(weights[i - window_width + 1:i + 1])
    return out


@nb.njit
def weighted_running_average_textbook(arr, weights):
    out = np.full(arr.shape, np.nan)
    weights_sum = 0
    mu_old = 0
    for i in range(len(arr)):
        weights_sum += weights[i]
        mu_old += weights[i] * arr[i]
        out[i] = mu_old / weights_sum
    return out, mu_old, weights_sum


# @nb.njit
# def weighted_moving_average_textbook(arr, weights, window_width):
#     out_running, mu_old, weights_sum = weighted_running_average_textbook(arr[:window_width], weights[:window_width])
#     out = np.full(arr.shape, np.nan)
#     out[:window_width] = out_running
#     for i in range(window_width, len(arr)):
#         weights_sum = weights_sum - weights[i - window_width] + weights[i]
#         mu_old = mu_old - arr[i - window_width] * weights[i - window_width] + arr[i] * weights[i]
#         out[i] = mu_old / weights_sum
#     return out



# @nb.njit
# def weighted_moving_average_textbook(arr, weights, window_width):
#     out = np.full(arr.shape, np.nan)
#     weights_sum = 0
#     mu_old = 0
#     for i in range(window_width):
#         weights_sum += weights[i]
#         mu_old += weights[i] * arr[i]
#         out[i] = mu_old / weights_sum
#     for i in range(window_width, len(arr)):
#         weights_sum = weights_sum - weights[i - window_width] + weights[i]
#         mu_old = mu_old - arr[i - window_width] * weights[i - window_width] + arr[i] * weights[i]
#         out[i] = mu_old / weights_sum
#     return out


@nb.njit
def weighted_moving_average_textbook(arr, weights, window_width):
    out = np.full(arr.shape, np.nan)
    W_sum_prev = np.sum(weights[:window_width])
    arr_times_W_sum_prev = np.sum(weights[:window_width] * arr[:window_width])
    out[window_width - 1] = arr_times_W_sum_prev / W_sum_prev
    for i in range(window_width, len(arr)):
        W_sum_prev = W_sum_prev - weights[i - window_width] + weights[i]
        arr_times_W_sum_prev = arr_times_W_sum_prev - arr[i - window_width] * weights[i - window_width] + arr[i] * weights[i]
        out[i] = arr_times_W_sum_prev / W_sum_prev
    return out



@nb.njit
def weighted_moving_standard_deviation_welford(arr, weights, window_width):
    out = np.full(arr.shape, np.nan)
    W_sum_prev = np.sum(weights[:window_width])
    arr_times_W_sum_prev = np.sum(weights[:window_width] * arr[:window_width])
    ss_prev = sum(weights[:window_width] * (arr[:window_width])**2) - arr_times_W_sum_prev**2 / W_sum_prev
    mu_prev = arr_times_W_sum_prev / W_sum_prev
    out[window_width - 1] = ss_prev
    for i in range(window_width, len(arr)):
        W_sum_next = W_sum_prev - weights[i - window_width] + weights[i]
        arr_times_W_sum_next = arr_times_W_sum_prev - arr[i - window_width] * weights[i - window_width] + arr[i] * weights[i]
        mu_next = arr_times_W_sum_next / W_sum_next
        ss_prev = ss_prev + weights[i] * (arr[i] - mu_prev) * (arr[i] - mu_next) - weights[i - window_width] * (arr[i - window_width] - mu_prev) * (arr[i - window_width] - mu_next)
        out[i] = ss_prev
        arr_times_W_sum_prev = arr_times_W_sum_next
        W_sum_prev = W_sum_next
        mu_prev = mu_next

    return np.sqrt(out / W_sum_prev)





In [162]:
from fast_borf.moving import move_std
import bottleneck as bn

In [163]:
LENGTH = 10
arr = np.arange(LENGTH)
arr = np.random.rand(LENGTH)
weights = np.ones(LENGTH)
window_width = 3

In [166]:
move_std(arr, window_width)

array([       nan,        nan, 0.28191131, 0.30372583, 0.11801701,
       0.15415849, 0.25806662, 0.26370085, 0.02691785, 0.18339841])

In [167]:
weighted_moving_standard_deviation_welford(arr, weights, window_width)

array([       nan,        nan, 0.28191131, 0.30372583, 0.11801701,
       0.15415849, 0.25806662, 0.26370085, 0.02691785, 0.18339841])

In [160]:
bn.move_std(arr, window_width)

array([       nan,        nan, 0.22265033, 0.23274135, 0.17682006,
       0.16921077, 0.07225643, 0.31407831, 0.28777185, 0.20885104])

In [159]:
weighted_moving_average_textbook(arr, weights, window_width)

array([       nan,        nan, 0.49035481, 0.43023078, 0.57272394,
       0.56502528, 0.72497727, 0.54926267, 0.44971061, 0.38345585])

In [158]:
bn.move_mean(arr, window_width)

array([       nan,        nan, 0.49035481, 0.43023078, 0.57272394,
       0.56502528, 0.72497727, 0.54926267, 0.44971061, 0.38345585])

In [129]:
W_sum_prev = np.sum(weights[:window_width])
arr_times_W_sum_prev = np.sum(weights[:window_width] * arr[:window_width])
ss_prev = sum(weights[:window_width] * (arr[:window_width])**2) - arr_times_W_sum_prev**2 / W_sum_prev

In [132]:
ss_prev

5.0

In [69]:
from fast_borf.moving import move_mean

In [92]:
import pandas as pd

In [112]:
LENGTH = 10000
arr = np.arange(LENGTH)
weights = np.ones(LENGTH)
window_width = 100

## Moving

### std

In [171]:
%%timeit
move_std(arr, window_width)

540 ns ± 2.12 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


### mean

In [113]:
%%timeit
weighted_moving_average_naive(arr, weights, window_width)

2.3 ms ± 18.3 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [114]:
%%timeit
weighted_moving_average_textbook(arr, weights, window_width)

20.5 μs ± 142 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [116]:
%%timeit
pd.Series(arr).rolling(window=window_width, min_periods=1).mean().values

117 μs ± 1.26 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [117]:
%%timeit
move_mean(arr.astype(float), window_width)

13 μs ± 62.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


## Running

In [98]:
weighted_running_average_textbook(arr, weights)[0]

array([  0. ,   0.5,   1. ,   1.5,   2. ,   2.5,   3. ,   3.5,   4. ,
         4.5,   5. ,   5.5,   6. ,   6.5,   7. ,   7.5,   8. ,   8.5,
         9. ,   9.5,  10. ,  10.5,  11. ,  11.5,  12. ,  12.5,  13. ,
        13.5,  14. ,  14.5,  15. ,  15.5,  16. ,  16.5,  17. ,  17.5,
        18. ,  18.5,  19. ,  19.5,  20. ,  20.5,  21. ,  21.5,  22. ,
        22.5,  23. ,  23.5,  24. ,  24.5,  25. ,  25.5,  26. ,  26.5,
        27. ,  27.5,  28. ,  28.5,  29. ,  29.5,  30. ,  30.5,  31. ,
        31.5,  32. ,  32.5,  33. ,  33.5,  34. ,  34.5,  35. ,  35.5,
        36. ,  36.5,  37. ,  37.5,  38. ,  38.5,  39. ,  39.5,  40. ,
        40.5,  41. ,  41.5,  42. ,  42.5,  43. ,  43.5,  44. ,  44.5,
        45. ,  45.5,  46. ,  46.5,  47. ,  47.5,  48. ,  48.5,  49. ,
        49.5,  50. ,  50.5,  51. ,  51.5,  52. ,  52.5,  53. ,  53.5,
        54. ,  54.5,  55. ,  55.5,  56. ,  56.5,  57. ,  57.5,  58. ,
        58.5,  59. ,  59.5,  60. ,  60.5,  61. ,  61.5,  62. ,  62.5,
        63. ,  63.5,

In [99]:
weighted_running_average_naive(arr, weights)[0]

array([  0. ,   0.5,   1. ,   1.5,   2. ,   2.5,   3. ,   3.5,   4. ,
         4.5,   5. ,   5.5,   6. ,   6.5,   7. ,   7.5,   8. ,   8.5,
         9. ,   9.5,  10. ,  10.5,  11. ,  11.5,  12. ,  12.5,  13. ,
        13.5,  14. ,  14.5,  15. ,  15.5,  16. ,  16.5,  17. ,  17.5,
        18. ,  18.5,  19. ,  19.5,  20. ,  20.5,  21. ,  21.5,  22. ,
        22.5,  23. ,  23.5,  24. ,  24.5,  25. ,  25.5,  26. ,  26.5,
        27. ,  27.5,  28. ,  28.5,  29. ,  29.5,  30. ,  30.5,  31. ,
        31.5,  32. ,  32.5,  33. ,  33.5,  34. ,  34.5,  35. ,  35.5,
        36. ,  36.5,  37. ,  37.5,  38. ,  38.5,  39. ,  39.5,  40. ,
        40.5,  41. ,  41.5,  42. ,  42.5,  43. ,  43.5,  44. ,  44.5,
        45. ,  45.5,  46. ,  46.5,  47. ,  47.5,  48. ,  48.5,  49. ,
        49.5,  50. ,  50.5,  51. ,  51.5,  52. ,  52.5,  53. ,  53.5,
        54. ,  54.5,  55. ,  55.5,  56. ,  56.5,  57. ,  57.5,  58. ,
        58.5,  59. ,  59.5,  60. ,  60.5,  61. ,  61.5,  62. ,  62.5,
        63. ,  63.5,

In [100]:
pd.Series(arr).expanding().mean().values

array([  0. ,   0.5,   1. ,   1.5,   2. ,   2.5,   3. ,   3.5,   4. ,
         4.5,   5. ,   5.5,   6. ,   6.5,   7. ,   7.5,   8. ,   8.5,
         9. ,   9.5,  10. ,  10.5,  11. ,  11.5,  12. ,  12.5,  13. ,
        13.5,  14. ,  14.5,  15. ,  15.5,  16. ,  16.5,  17. ,  17.5,
        18. ,  18.5,  19. ,  19.5,  20. ,  20.5,  21. ,  21.5,  22. ,
        22.5,  23. ,  23.5,  24. ,  24.5,  25. ,  25.5,  26. ,  26.5,
        27. ,  27.5,  28. ,  28.5,  29. ,  29.5,  30. ,  30.5,  31. ,
        31.5,  32. ,  32.5,  33. ,  33.5,  34. ,  34.5,  35. ,  35.5,
        36. ,  36.5,  37. ,  37.5,  38. ,  38.5,  39. ,  39.5,  40. ,
        40.5,  41. ,  41.5,  42. ,  42.5,  43. ,  43.5,  44. ,  44.5,
        45. ,  45.5,  46. ,  46.5,  47. ,  47.5,  48. ,  48.5,  49. ,
        49.5,  50. ,  50.5,  51. ,  51.5,  52. ,  52.5,  53. ,  53.5,
        54. ,  54.5,  55. ,  55.5,  56. ,  56.5,  57. ,  57.5,  58. ,
        58.5,  59. ,  59.5,  60. ,  60.5,  61. ,  61.5,  62. ,  62.5,
        63. ,  63.5,